# pandaN.h5 Analysis Notebook

In [ ]:
# parameters
xname = "inenc-1-val_dataset"
yname = "fmc_in-val1_dataset"
tiled_id = ""

In [ ]:
# imports:
import numpy as np
import json
import matplotlib.pyplot as plt
import h5py
from scipy.signal import savgol_filter
from xas_toolbox.utils.maths.start_stop import _get_start_stop

In [ ]:
# artifacts:
ARTIFACTS = {
    "notebook": "/tmp/notebook.html",
    "plot": "/tmp/plot_result.png",
    "report": "/tmp/report.json"
    }

In [ ]:
# read dataset and get (x, y) data:
fpath = "/data/data.h5"

with h5py.File(fpath, "r") as f:
        if not yname in f.keys() or not yname in f.keys():
            raise AttributeError("x / y axis not found in scan.")
        x: np.ndarray = f[xname][...]
        y: np.ndarray = f[yname][...]


## Utils for getting number of repetitions:

In [ ]:
def find_turning_points(arr:np.ndarray, min_spacing:int=None):
    if min_spacing is None: min_spacing = int(len(arr)/20)

    arrn = (arr-np.min(arr))/(np.max(arr)-np.min(arr))
    arrn = np.round(arrn, 0)
    wheremax = np.where(arrn == np.max(arrn))[0]
    ss = _get_start_stop(wheremax, np.diff(wheremax), min_spacing)
    mean_pos = [np.mean(s, dtype=int) for s in ss]
    return mean_pos

## Utils for scan diagnostics:

In [ ]:
def get_scan_metrics(y:np.ndarray)->dict:

    yf = savgol_filter(y, int(len(y)/10), 2)
    noise = np.std(y-yf)

    res = {"min_y": np.min(y), "max_y": np.max(y),
           "noise": noise}
    return res

def repetition_processing(x:np.ndarray, y:np.ndarray, min_spacing:int=None):
    """
    detect number of repetitions of scan.
    """
    mean_peaks_y = find_turning_points(y, min_spacing)
    mean_peaks_x = find_turning_points(np.abs(np.diff(x, 2)), min_spacing)

    nreps = np.min([len(mean_peaks_x)+1, len(mean_peaks_y)])
    print(f"Found {nreps} repetitions.")
    report = {}

    start = 0; stop = np.concat((mean_peaks_x, [len(x)]))

    for i in range(nreps):

        res = get_scan_metrics(y[start:stop[i]])
        res["bounds"] = (start, stop[i])
        report[f"Repetition {i+1}"] = res
        
        start = stop[i]

    # divide data evenly and do snr analysis:
    y2 = y.copy()
    while len(y2) % nreps != 0:
        # pad the end-point of the dataset to get even splitting.
        y2 = np.concat((y2, [y2[-1]]*len(y2)%nreps))

    ysplit = np.vstack((np.split(y2, nreps)))
    cy = np.cumsum(ysplit, axis=0)/np.sum(ysplit, axis=0)
    noise = np.std(cy - savgol_filter(cy, int(len(y)/10), 2), axis=1)

    
    return report, noise

In [ ]:
int(0.02*len(y))

In [ ]:
report, noise = repetition_processing(x, y)

In [ ]:
report["snr"] = noise

with open(ARTIFACTS["report"], "w"):
    json.dumps(report, indent=4, default=str)

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(10, 5), layout="tight")
xplot = np.linspace(1, len(noise)+1, len(noise))

fig.suptitle(f"Data for: {fpath.split('/')[-1]}\ntiled id: {tiled_id}")

ax[0, 0].set_title("xy data")
ax[0, 0].plot(x, y, lw=1)

ax[0, 1].set_title(r"$\frac{noise}{\sqrt{N}}$")
ax[0, 1].plot(xplot, noise/np.sqrt(xplot), "o-");
ax[0, 1].set_xlabel("Repetition No.")
for k, v in report.items():
    if isinstance(v, dict):
        bounds = v["bounds"]
        x_tmp =  x[bounds[0]:bounds[-1]]
        y_tmp = y[bounds[0]:bounds[-1]]
        ax[1, 0].plot(x_tmp, y_tmp, label=k, lw=1, alpha=0.8)
        ax[1, 1].plot(k, v["noise"], "o")


[a.set_xlabel(xname) for a in [ax[0, 0], ax[1, 0]]]
[a.set_ylabel(yname) for a in [ax[0, 0], ax[1, 0]]]

[a.set_ylabel("Estimated Noise") for a in [ax[0, 1], ax[1, 1]]]

ax[1, 0].legend()

fig.savefig(ARTIFACTS["plot"])
plt.show();